# Smart MCQ Solver

Predicts the **top 3 answers** for each 5-option multiple-choice question. Metric: MAP@3.

### Required components
| requirement | component | section |
|---|---|---|
| **Pretrained model** | `microsoft/deberta-v3-large` fine-tuned as a 5-way multiple-choice head (5-fold) | 11b |
| **Model from scratch** | Two-layer MLP in pure NumPy — used both for the similarity ranker and the answer-artifact discriminator | 10, 11c |
| **Model of choice** | Calibrated score fusion over 10 signals, similarity-gated, with random-search weights | 12 |

### All signals
| name | kind | learns from |
|---|---|---|
| `tfidf` | corpus lookup | — |
| `st` | corpus lookup (`BAAI/bge-large-en-v1.5`) | — |
| `rag` | lookup + retrieved context | — |
| `mlp` | NumPy MLP over similarity features | 2000 questions |
| `nli` | cross-encoder entailment − contradiction | — (zero-shot) |
| `llm` | few-shot letter logits (`Qwen2.5` / `Phi-3.5`) | — (zero-shot) |
| `struct` | NumPy MLP over structural features | 2000 questions |
| `prior` | answer-letter frequency prior | 2000 questions |
| `mcq` | **fine-tuned DeBERTa-v3 MCQ head, 5-fold ensemble** | 2000 questions |
| `disc` | **answer-artifact discriminator, NumPy MLP** | **10,000 option pairs** |

### The core problem
The lookup signals (`tfidf`, `st`, `rag`, `mlp`) answer by matching options against
the training corpus. They score ~0.99 wherever a question near-duplicates a training
row and contribute nothing on genuinely novel questions — which inflates validation
far above the hidden test set. Tables therefore report **EASY** and **HARD** columns
separately (chance = 0.3667), and weights are tuned per bucket.

`disc` is the one signal that is **question-independent**: it learns what separates a
real answer from a generated distractor, so it transfers to questions with no
near-duplicate anywhere in training. Its OOF MAP@3 in Section 11c is the single most
informative number in the notebook.


## 1. Setup

In [1]:
# Install required packages
!pip install -q wandb sentence-transformers transformers accelerate

import os
import re
import json
import warnings

import numpy as np
import pandas as pd
import torch
import wandb

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")
print(f"Pandas: {pd.__version__}")
print(f"NumPy : {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [2]:
from kaggle_secrets import UserSecretsClient

# Login to Weights & Biases
secrets = UserSecretsClient()
WANDB_KEY = secrets.get_secret("WandB_API_key")

wandb.login(key=WANDB_KEY, relogin=True)

print("W&B login successful.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: soumyaranjanpanda01 (23f2004742-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful.


In [3]:
# Paths
BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"

TRAIN_PATH = f"{BASE}/train.csv"
TEST_PATH = f"{BASE}/test.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"

# Constants
OPTIONS = ["A", "B", "C", "D", "E"]
SEED = 42
VAL_SIZE = 0.20

WANDB_PROJECT = "23f2004742-t22026"

## 2. Load Dataset

In [4]:
# Load datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train Shape : {train_df.shape}")
print(f"Test Shape  : {test_df.shape}")

print("\nTraining Answer Distribution")
print(train_df["answer"].value_counts().sort_index())

train_df.head(3)

Train Shape : (2000, 8)
Test Shape  : (500, 7)

Training Answer Distribution
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
# Quick dataset overview

# Option length statistics
for opt in OPTIONS:
    lengths = train_df[opt].str.len()
    print(f"{opt}: Mean = {lengths.mean():.0f}, Max = {lengths.max()}")

# Duplicate prompts
prompt_counts = train_df["prompt"].value_counts()

print(f"\nUnique Prompts : {len(prompt_counts)}")
print(f"Repeated Prompts: {(prompt_counts > 1).sum()}")

# Question pattern
def get_style(prompt):
    prompt = prompt.lower()

    if "pick the best" in prompt:
        return "Pick the best"
    if "choose the correct" in prompt:
        return "Choose the correct"
    if "which of the following" in prompt:
        return "Which of the following"

    return "Other"

print("\nQuestion Styles")
print(train_df["prompt"].apply(get_style).value_counts())

A: Mean = 164, Max = 472
B: Mean = 167, Max = 662
C: Mean = 167, Max = 530
D: Mean = 163, Max = 450
E: Mean = 164, Max = 587

Unique Prompts : 1758
Repeated Prompts: 212

Question Styles
prompt
Other                     1063
Which of the following     329
Pick the best              316
Choose the correct         292
Name: count, dtype: int64


## 3. Train / Validation Split

Split the dataset by **prompt** so that the same question never appears in both training and validation sets.

In [ ]:
# Split unique prompts
unique_prompts = train_df["prompt"].unique()

train_prompts, val_prompts = train_test_split(
    unique_prompts, test_size=VAL_SIZE, random_state=SEED
)

train_split = train_df[train_df["prompt"].isin(train_prompts)].reset_index(drop=True)
val_split   = train_df[train_df["prompt"].isin(val_prompts)].reset_index(drop=True)

overlap = set(train_split["prompt"]) & set(val_split["prompt"])
assert len(overlap) == 0


def max_similarity_to(query_prompts, reference_prompts) -> np.ndarray:
    """Max TF-IDF cosine similarity of each query prompt to ANY reference prompt.

    Defined once and used for BOTH val and test so the threshold means the same
    thing on each side. It must be applied to CLEANED text in both cases: the
    boilerplate prefixes ("Pick the best possible answer:") are shared by every
    raw prompt and inflate cosine similarity, so measuring val on raw text and
    test on cleaned text would make the two incomparable. The actual call is
    therefore deferred to the preprocessing cell below, after cleaning.
    """
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    mat = vec.fit_transform(list(reference_prompts) + list(query_prompts))
    ref = mat[:len(reference_prompts)]
    qry = mat[len(reference_prompts):]
    return cosine_similarity(qry, ref).max(axis=1)


# A question is EASY if some training prompt closely resembles it (answerable by
# lookup) and HARD if none does (only reasoning can answer it).
HARD_THRESHOLD = 0.7

print(f"Train : {len(train_split)} rows ({len(train_split)/len(train_df):.1%})")
print(f"Valid : {len(val_split)} rows ({len(val_split)/len(train_df):.1%})")
print(f"Exact prompt overlap : {len(overlap)}")


## 4. Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    """Clean prompts and options."""

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    boilerplate = [
        r"^Pick the best possible answer:\s*",
        r"^Choose the correct answer:\s*",
        r"^Select the most accurate option:\s*",
        r"^Identify the correct statement:\s*",
        r"^Determine the correct option:\s*",
        r"\s*among the listed options\.?$",
        r"\s*from the following choices\.?$",
        r"\s*based on the given context\.?$",
        r"\s*carefully\.?$",
    ]

    for pattern in boilerplate:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    return text


def preprocess_row(row):
    """Return a cleaned question."""

    return {
        "prompt": clean_text(row["prompt"]),
        "A": clean_text(row["A"]),
        "B": clean_text(row["B"]),
        "C": clean_text(row["C"]),
        "D": clean_text(row["D"]),
        "E": clean_text(row["E"]),
    }


# Apply preprocessing
# train_df_clean is essential: Section 14 builds the test-time indexes from the
# FULL training set, and previously used the RAW train_df while querying with
# CLEANED test rows. That meant the index still contained boilerplate prefixes
# ("Pick the best possible answer:") that had been stripped from every query --
# a preprocessing mismatch that existed at test time but never during
# validation, so it was invisible in the val scores.
train_split   = train_split.copy()
val_split     = val_split.copy()
test_df_clean = test_df.copy()
train_df_clean = train_df.copy()

for df in [train_split, val_split, test_df_clean, train_df_clean]:
    df["prompt"] = df["prompt"].apply(clean_text)

    for opt in OPTIONS:
        df[opt] = df[opt].apply(clean_text)

print("Text preprocessing completed.")

raw = train_df.iloc[0]["prompt"]
clean = train_split.iloc[0]["prompt"]

print("\nBefore:", raw[:100])
print("After :", clean[:100])

# ---------------------------------------------------------------------------
# Similarity of each val prompt to the training prompts, computed on CLEANED
# text so it is directly comparable with the test-set figure in Section 14.
# (Computing this before cleaning inflated it, because every raw prompt shares
# the same boilerplate prefix.)
# ---------------------------------------------------------------------------
_sim = max_similarity_to(
    val_split["prompt"].unique().tolist(),
    train_split["prompt"].unique().tolist(),
)
val_sim_to_train = val_split["prompt"].map(
    dict(zip(val_split["prompt"].unique().tolist(), _sim))
).values
val_hard_mask = val_sim_to_train < HARD_THRESHOLD

print(f"\nEASY val rows (near-duplicate of a train prompt) : {(~val_hard_mask).sum()}")
print(f"HARD val rows (no close neighbour)               : {val_hard_mask.sum()}")

## 5. Evaluation Metric (MAP@3)

In [ ]:
def average_precision_at_3(actual, predicted):
    """Compute AP@3 for one prediction."""

    predicted = [str(p).strip().upper() for p in predicted[:3]]

    for rank, pred in enumerate(predicted, start=1):
        if pred == actual:
            return 1 / rank

    return 0.0


def map_at_3(actuals, predictions):
    """Compute Mean Average Precision@3."""

    assert len(actuals) == len(predictions)

    scores = [
        average_precision_at_3(a, p)
        for a, p in zip(actuals, predictions)
    ]

    return float(np.mean(scores))


# Quick verification
assert average_precision_at_3("A", ["A", "B", "C"]) == 1
assert average_precision_at_3("A", ["B", "A", "C"]) == 0.5
assert abs(average_precision_at_3("A", ["C", "D", "A"]) - 1 / 3) < 1e-9
assert average_precision_at_3("A", ["B", "C", "D"]) == 0

print("MAP@3 verified ✓")


# ---------------------------------------------------------------------------
# MAP@3 is the leaderboard metric, but the project brief also asks for accuracy
# and F1. Both are derived from the TOP-1 label of the ranked prediction, so one
# helper keeps every section reporting the same quantities the same way.
#
# Macro F1 weights each option equally, so it exposes a model that scores well by
# favouring the frequent letters (B 24.5%, C 23.0%) while failing on the rare
# ones (D, E) -- something neither MAP@3 nor plain accuracy would reveal.
# ---------------------------------------------------------------------------
from sklearn.metrics import accuracy_score, f1_score, classification_report


def eval_metrics(actuals, predictions) -> dict:
    """MAP@3 + accuracy + macro/weighted F1 from ranked top-3 predictions."""
    actuals = [str(a).strip().upper() for a in actuals]
    top1 = [str(p[0]).strip().upper() for p in predictions]
    return {
        "map3":        map_at_3(actuals, predictions),
        "accuracy":    float(accuracy_score(actuals, top1)),
        "f1_macro":    float(f1_score(actuals, top1, labels=OPTIONS,
                                      average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(actuals, top1, labels=OPTIONS,
                                      average="weighted", zero_division=0)),
    }


def report_metrics(name, actuals, predictions, extra=None) -> dict:
    """Print a one-line summary and return the dict for wandb."""
    m = eval_metrics(actuals, predictions)
    if extra:
        m.update(extra)
    print(f"{name:<26} MAP@3 {m['map3']:.4f} | Acc {m['accuracy']:.4f} | "
          f"Macro-F1 {m['f1_macro']:.4f} | Wgt-F1 {m['f1_weighted']:.4f}")
    return m


# Self-test. Note macro F1 averages over ALL five labels, so a perfect run that
# never uses D or E scores (1+1+1+0+0)/5 = 0.6 rather than 1.0 -- which is the
# point of tracking it: unused or never-predicted options are penalised.
_p3 = eval_metrics(["A", "B", "C"], [["A","B","C"], ["B","A","C"], ["C","A","B"]])
assert _p3["map3"] == 1.0 and _p3["accuracy"] == 1.0
assert abs(_p3["f1_macro"] - 0.6) < 1e-9, _p3["f1_macro"]

_all5 = eval_metrics(OPTIONS, [[o] + [x for x in OPTIONS if x != o] for o in OPTIONS])
assert _all5["map3"] == 1.0 and _all5["accuracy"] == 1.0 and _all5["f1_macro"] == 1.0

_half = eval_metrics(["A", "B"], [["B","A","C"], ["B","A","C"]])
assert _half["accuracy"] == 0.5 and abs(_half["map3"] - 0.75) < 1e-9

print("Metric helpers ready (MAP@3, accuracy, macro F1, weighted F1)")


## 6. TF-IDF Baseline

Create a TF-IDF index using the training set's correct answers and rank each option by cosine similarity.

In [ ]:
# Build TF-IDF index
correct_texts_train = [
    f"{row['prompt']} {row[row['answer']]}"
    for _, row in train_split.iterrows()
]

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=80000,
    strip_accents="unicode",
    min_df=1,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(correct_texts_train)

print(f"Vocabulary Size : {len(tfidf_vectorizer.vocabulary_)}")
print(f"Matrix Shape    : {tfidf_matrix.shape}")

In [ ]:
def tfidf_rank_options(row):
    """Rank answer options using TF-IDF cosine similarity."""

    scores = {}

    for opt in OPTIONS:
        query = f"{row['prompt']} {row[opt]}"
        query_vec = tfidf_vectorizer.transform([query])

        scores[opt] = float(
            cosine_similarity(query_vec, tfidf_matrix).max()
        )

    return sorted(scores, key=lambda x: -scores[x])


# Validation
tfidf_predictions = [
    tfidf_rank_options(row)[:3]
    for _, row in val_split.iterrows()
]

tfidf_map3 = map_at_3(
    val_split["answer"].tolist(),
    tfidf_predictions,
)

top1 = sum(
    pred[0] == ans
    for pred, ans in zip(tfidf_predictions, val_split["answer"])
)

print(f"MAP@3   : {tfidf_map3:.4f}")
print(f"Top-1   : {top1}/{len(val_split)} ({top1/len(val_split):.2%})")

In [ ]:
# Log TF-IDF results
wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-baseline",
    config={
        "model": "TF-IDF Cosine Similarity",
        "ngram_range": "(1,2)",
        "max_features": 80000,
        "sublinear_tf": True,
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

_m = report_metrics("TF-IDF", val_split["answer"].tolist(), tfidf_predictions)
wandb.log({f"val_{k}": v for k, v in _m.items()})

wandb.finish()

print("TF-IDF results logged.")

## 7. Sentence Transformer

Use a pretrained sentence embedding model to compare the semantic similarity between questions and answer options.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading model...")

# Upgraded from all-MiniLM-L6-v2 → BAAI/bge-large-en-v1.5
# bge-large consistently tops MTEB leaderboard and significantly
# outperforms MiniLM on semantic similarity tasks.
# Single source of truth: the wandb configs below reference this variable, so a
# future model swap cannot leave the logged name stale (it previously still read
# "all-MiniLM-L6-v2" here long after the loader moved to bge-large).
ST_MODEL_ID = "BAAI/bge-large-en-v1.5"
st_model = SentenceTransformer(ST_MODEL_ID)

print("Model loaded.")
print(f"Embedding Size : {st_model.get_sentence_embedding_dimension()}")


In [ ]:
# Encode training corpus
train_corpus = [
    f"{row['prompt']} {row[row['answer']]}"
    for _, row in train_split.iterrows()
]

print("Encoding training corpus...")

train_embeddings = st_model.encode(
    train_corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

train_embeddings = normalize(train_embeddings)

print(f"Embedding Matrix : {train_embeddings.shape}")

In [ ]:
def st_rank_options(row: pd.Series) -> list:
    """Rank answer options using sentence embedding similarity."""

    option_texts = [
        f"{row['prompt']} {row[opt]}"
        for opt in OPTIONS
    ]

    option_embs = st_model.encode(
        option_texts,
        convert_to_numpy=True
    )
    option_embs = normalize(option_embs)

    scores = (option_embs @ train_embeddings.T).max(axis=1)

    return [OPTIONS[i] for i in np.argsort(-scores)]


print("Evaluating on validation set...")

st_predictions = [
    st_rank_options(row)[:3]
    for _, row in val_split.iterrows()
]

st_map3 = map_at_3(
    val_split["answer"].tolist(),
    st_predictions
)

top1_st = sum(
    pred[0] == ans
    for pred, ans in zip(st_predictions, val_split["answer"])
)

print(f"MAP@3 : {st_map3:.4f}")
print(f"Top-1 : {top1_st}/{len(val_split)} ({top1_st/len(val_split):.2%})")

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="sentence-transformer",
    config={
        "model": ST_MODEL_ID,
        "embedding_dim": st_model.get_sentence_embedding_dimension(),
        "similarity": "cosine",
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

_m = report_metrics("Sentence Transformer", val_split["answer"].tolist(), st_predictions)
wandb.log({f"val_{k}": v for k, v in _m.items()})

wandb.finish()

print("Sentence Transformer results logged.")

## 8. NLI Cross-Encoder Scoring

Score each `(question, option)` pair **directly** with an NLI cross-encoder and
rank by `entailment - contradiction`.

The previous version fed each answer option to `pipeline("zero-shot-classification")`
as a *candidate label*, which wraps it in the template `"This example is {option}."`.
For full-sentence options that produces a meaningless hypothesis, which is why it
scored 0.5339 -- barely above the 0.3667 chance baseline for MAP@3. Using the
cross-encoder directly on the pair is the intended use of this model.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading NLI cross-encoder...")

nli_model_id  = "cross-encoder/nli-deberta-v3-large"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_id)
nli_model     = AutoModelForSequenceClassification.from_pretrained(nli_model_id)
nli_model.eval()

# Kept in fp32: DeBERTa-v3 is known to overflow in fp16. It is only ~1.7GB,
# which leaves plenty of room alongside Phi-3.5-mini later.
if torch.cuda.is_available():
    nli_model = nli_model.cuda()

# Resolve label positions from the config rather than hardcoding indices,
# since label order differs between NLI checkpoints.
_id2label   = {int(k): str(v).lower() for k, v in nli_model.config.id2label.items()}
ENTAIL_IDX  = next(i for i, l in _id2label.items() if "entail"  in l)
CONTRA_IDX  = next(i for i, l in _id2label.items() if "contra"  in l)

print("Model loaded.")
print(f"Labels: {_id2label} | entailment={ENTAIL_IDX} contradiction={CONTRA_IDX}")


In [ ]:
@torch.no_grad()
def scores_nli(row) -> np.ndarray:
    """Score all 5 options at once: entailment - contradiction logit."""

    premises   = [str(row["prompt"])] * len(OPTIONS)
    hypotheses = [str(row[opt]) for opt in OPTIONS]

    enc = nli_tokenizer(
        premises, hypotheses,
        return_tensors="pt", padding=True, truncation=True, max_length=512,
    ).to(nli_model.device)

    logits = nli_model(**enc).logits.float()
    return (logits[:, ENTAIL_IDX] - logits[:, CONTRA_IDX]).cpu().numpy().astype(np.float32)


def nli_rank_options(row) -> list:
    s = scores_nli(row)
    return [OPTIONS[i] for i in np.argsort(-s)]


print("Evaluating NLI on validation set...")

nli_scores_val = [scores_nli(row) for _, row in val_split.iterrows()]
nli_predictions = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in nli_scores_val]

nli_map3 = map_at_3(val_split["answer"].tolist(), nli_predictions)
top1_nli = sum(p[0] == a for p, a in zip(nli_predictions, val_split["answer"]))

print(f"MAP@3 : {nli_map3:.4f}")
print(f"Top-1 : {top1_nli}/{len(val_split)} ({top1_nli/len(val_split):.2%})")


In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="zero-shot-nli",
    config={
        "model": nli_model_id,
        "approach": "cross-encoder entailment minus contradiction",
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

_m = report_metrics("NLI cross-encoder", val_split["answer"].tolist(), nli_predictions)
wandb.log({f"val_{k}": v for k, v in _m.items()})

wandb.finish()

print("Zero-shot NLI results logged.")

## 9. Retrieval-Augmented Ranking (RAG)

Retrieve the most similar training questions and use their correct answers as additional context when ranking the options.

In [ ]:
def build_rag_index(train_df: pd.DataFrame, encoder):
    """Build embeddings for training prompts."""

    prompts = train_df["prompt"].tolist()
    correct_answers = [
        row[row["answer"]]
        for _, row in train_df.iterrows()
    ]

    prompt_embs = encoder.encode(
        prompts,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    prompt_embs = normalize(prompt_embs)

    return prompt_embs, correct_answers


print("Building retrieval index...")

rag_prompt_embs, rag_correct_texts = build_rag_index(
    train_split,
    st_model,
)

print(f"Index Size : {len(rag_prompt_embs)}")

In [ ]:
def rag_rank_options(row: pd.Series, k: int = 5) -> list:
    """Rank answer options using retrieval-augmented similarity."""
    
    # Retrieve similar questions
    q_emb = st_model.encode([row['prompt']], convert_to_numpy=True)
    q_emb = normalize(q_emb)
    retrieval_scores = (q_emb @ rag_prompt_embs.T)[0]
    top_k_idx = np.argsort(-retrieval_scores)[:k]

    # Build retrieval context
    context_texts = [rag_correct_texts[i] for i in top_k_idx]
    context_embs  = st_model.encode(context_texts, convert_to_numpy=True)
    context_embs  = normalize(context_embs)

    # Score answer options
    option_texts = [f"{row['prompt']} {row[opt]}" for opt in OPTIONS]
    option_embs  = normalize(st_model.encode(option_texts, convert_to_numpy=True))

    sim_corpus  = (option_embs @ train_embeddings.T).max(axis=1)
    sim_context = (option_embs @ context_embs.T).max(axis=1)

    # Combine: give slightly more weight to retrieved context
    combined = 0.6 * sim_corpus + 0.4 * sim_context
    return [OPTIONS[i] for i in np.argsort(-combined)]


print("Evaluating RAG on validation set...")
rag_predictions = [rag_rank_options(row)[:3] for _, row in val_split.iterrows()]
rag_map3 = map_at_3(val_split['answer'].tolist(), rag_predictions)

top1_rag = sum(p[0] == a for p, a in zip(rag_predictions, val_split['answer']))
print(f"MAP@3 : {rag_map3:.4f}")
print(f"Top-1 : {top1_rag}/{len(val_split)} ({top1_rag/len(val_split):.2%})")

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="rag-retrieval",
    config={
        "model": ST_MODEL_ID,   # RAG retrieves with the same encoder
        "approach": "RAG",
        "retrieval_k": 5,
        "corpus_weight": 0.6,
        "context_weight": 0.4,
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

_m = report_metrics("RAG", val_split["answer"].tolist(), rag_predictions)
wandb.log({f"val_{k}": v for k, v in _m.items()})

wandb.finish()

print("RAG results logged.")

## 10. MLP + Structural Artifact Ranker

Two learned rankers over handcrafted features:

**`mlp`** — the original question-level MLP over TF-IDF corpus similarities.

**`struct`** — a per-option ranker over *label-free structural* features. The options
in this dataset are not independent: the distractors read as systematic rewordings
of one base statement (inserted negations, swapped clauses, shuffled qualifiers).
That is a property of how the data was generated, so it can be learned from the
training labels and applied to any question -- including questions with no
near-duplicate in the training corpus, which is precisely where every
similarity-based model degrades to guessing.

Features are all computable without labels: length and its rank within the
question, negation-word counts, option-to-option token overlap (mean/max/min and
near-duplicate counts), overlap with the question, digit/stopword/unique-token
ratios, the corpus similarity and its gap to the best option, and the option
letter itself (which lets the model absorb the answer-position prior -- the
training labels are not uniform, and predicting the three most common letters
alone scores 0.4213 versus 0.3667 for chance).

A caveat worth stating: this exploits generation artifacts rather than
understanding, so it only transfers if the test split was generated the same way
as the training split.


In [ ]:
# ---------------------------------------------------------------------------
# Original question-level TF-IDF features (unchanged)
# ---------------------------------------------------------------------------
def extract_features(df: pd.DataFrame, vec, mat) -> tuple:
    """Extract TF-IDF similarity features for the MLP."""
    X, y = [], []
    for _, row in df.iterrows():
        sims = []
        for opt in OPTIONS:
            q = f"{row['prompt']} {row[opt]}"
            v = vec.transform([q])
            sims.append(float(cosine_similarity(v, mat).max()))

        diffs = [sims[i] - sims[j] for i in range(5) for j in range(i + 1, 5)]
        X.append(sims + diffs)
        if 'answer' in df.columns:
            y.append(OPTIONS.index(row['answer']))

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


# ---------------------------------------------------------------------------
# Structural (label-free) per-option features
# ---------------------------------------------------------------------------
NEG_WORDS = {
    "not", "no", "never", "cannot", "cant", "doesnt", "dont", "isnt", "arent",
    "wasnt", "werent", "none", "neither", "nor", "without", "fails", "failed",
    "incorrect", "false", "untrue", "unable", "lacks", "lacking", "denies",
    "denied", "rejects", "rejected", "opposite", "contrary",
}
STOPWORDS = {
    "the", "a", "an", "of", "to", "in", "is", "are", "was", "were", "be", "been",
    "that", "this", "it", "as", "for", "with", "on", "by", "at", "from", "and",
    "or", "but", "which", "who", "whom", "his", "her", "its", "their", "they",
}


def _toks(s):
    return re.findall(r"[a-z0-9]+", str(s).lower())


def _z(a):
    a = np.asarray(a, dtype=np.float64)
    sd = a.std()
    return (a - a.mean()) / sd if sd > 1e-8 else np.zeros_like(a)


def _rank01(a):
    a = np.asarray(a, dtype=np.float64)
    return np.argsort(np.argsort(a)).astype(np.float64) / (len(a) - 1)


def _jaccard(x, y):
    u = len(x | y)
    return len(x & y) / u if u else 0.0


def structural_features(row, corpus_sims=None) -> np.ndarray:
    """Label-free per-option features -> (5, n_feat).

    Describes how each option relates to its SIBLINGS rather than what it says,
    so it keeps working where corpus lookup fails.
    """
    texts = [str(row[o]) for o in OPTIONS]
    toks  = [set(_toks(t)) for t in texts]
    ptoks = set(_toks(row["prompt"]))
    n = len(OPTIONS)

    lens_c = np.array([len(t) for t in texts], dtype=np.float64)
    lens_w = np.array([len(_toks(t)) for t in texts], dtype=np.float64)
    negs   = np.array([sum(1 for w in _toks(t) if w in NEG_WORDS) for t in texts], dtype=np.float64)
    digits = np.array([sum(ch.isdigit() for ch in t) for t in texts], dtype=np.float64)
    stopr  = np.array([sum(1 for w in _toks(t) if w in STOPWORDS) / max(len(_toks(t)), 1)
                       for t in texts], dtype=np.float64)
    uniqr  = np.array([len(set(_toks(t))) / max(len(_toks(t)), 1) for t in texts], dtype=np.float64)
    jac_p  = np.array([_jaccard(tk, ptoks) for tk in toks], dtype=np.float64)

    # option-to-option overlap: rewordings of a shared base statement leave a
    # distinctive signature here
    sim_oo = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                sim_oo[i, j] = _jaccard(toks[i], toks[j])
    off = [[j for j in range(n) if j != i] for i in range(n)]
    mean_oo = np.array([sim_oo[i, off[i]].mean() for i in range(n)])
    max_oo  = np.array([sim_oo[i, off[i]].max()  for i in range(n)])
    min_oo  = np.array([sim_oo[i, off[i]].min()  for i in range(n)])
    n_dup   = np.array([(sim_oo[i, off[i]] > 0.8).sum() for i in range(n)], dtype=np.float64)

    cs = np.zeros(n) if corpus_sims is None else np.asarray(corpus_sims, dtype=np.float64)

    feats = [
        _z(lens_c), _rank01(lens_c),
        _z(lens_w),
        _z(negs), (negs > 0).astype(np.float64),
        _z(jac_p), _rank01(jac_p),
        _z(mean_oo), _z(max_oo), _z(min_oo), n_dup,
        _z(digits), _z(stopr), _z(uniqr),
        _z(cs), _rank01(cs), cs - cs.max(),
        (lens_c == lens_c.max()).astype(np.float64),
        (lens_c == lens_c.min()).astype(np.float64),
    ]
    F = np.stack(feats, axis=1)
    # option letter -> lets the ranker absorb the answer-position prior
    F = np.concatenate([F, np.eye(n)], axis=1)
    return F.astype(np.float32)


N_STRUCT_FEATS = structural_features(
    {"prompt": "x", **{o: "y" for o in OPTIONS}}).shape[1]


def build_struct_dataset(df, vec, mat):
    """Pointwise dataset: one example PER OPTION (5x the rows, more data-efficient
    than a listwise model on ~1.6k questions)."""
    Xs, ys = [], []
    for _, row in df.iterrows():
        sims = [float(cosine_similarity(vec.transform([f"{row['prompt']} {row[o]}"]), mat).max())
                for o in OPTIONS]
        Xs.append(structural_features(row, corpus_sims=sims))
        if "answer" in df.columns:
            ys.extend(1 if o == row["answer"] else 0 for o in OPTIONS)
    X = np.concatenate(Xs, axis=0).astype(np.float32)
    return X, np.array(ys, dtype=np.int32)


def label_log_prior(df) -> np.ndarray:
    """log P(letter) from training labels; a tunable tie-break in the ensemble."""
    c = Counter(df["answer"])
    tot = sum(c.values())
    return np.array([np.log((c.get(o, 0) + 1) / (tot + len(OPTIONS))) for o in OPTIONS],
                    dtype=np.float32)


print("Extracting features for train split...")
X_train, y_train = extract_features(train_split, tfidf_vectorizer, tfidf_matrix)
print("Extracting features for val split...")
X_val, y_val = extract_features(val_split, tfidf_vectorizer, tfidf_matrix)

print("Building structural dataset...")
S_train, sy_train = build_struct_dataset(train_split, tfidf_vectorizer, tfidf_matrix)

LABEL_LOGPRIOR = label_log_prior(train_split)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"S_train : {S_train.shape}  ({N_STRUCT_FEATS} structural features)")
print(f"label prior : {dict(zip(OPTIONS, np.round(np.exp(LABEL_LOGPRIOR), 3)))}")


In [ ]:
#  MLP in pure NumPy
class MLP:
    """Two-layer MLP implemented in NumPy."""
    
    def __init__(self, n_in, n_hidden, n_out, lr=0.05, seed=42):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / n_in)   # He initialisation
        self.W1 = rng.normal(0, scale, (n_in, n_hidden)).astype(np.float32)
        self.b1 = np.zeros(n_hidden, dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2.0 / n_hidden), (n_hidden, n_out)).astype(np.float32)
        self.b2 = np.zeros(n_out, dtype=np.float32)
        self.lr = lr

    def _relu(self, x):
        return np.maximum(0, x)

    def _softmax(self, x):
        ex = np.exp(x - x.max(axis=1, keepdims=True))
        return ex / ex.sum(axis=1, keepdims=True)

    def forward(self, X):
        self.h1  = self._relu(X @ self.W1 + self.b1)
        self.out = self._softmax(self.h1 @ self.W2 + self.b2)
        return self.out

    def backward(self, X, y_true):
        n = len(X)
        # Cross-entropy gradient w.r.t. softmax output
        d_out = self.out.copy()
        d_out[np.arange(n), y_true] -= 1
        d_out /= n
        # Layer 2 gradients
        dW2 = self.h1.T @ d_out
        db2 = d_out.sum(axis=0)
        # Layer 1 gradients (ReLU derivative = 1 if h1 > 0)
        d_h1 = (d_out @ self.W2.T) * (self.h1 > 0)
        dW1  = X.T @ d_h1
        db1  = d_h1.sum(axis=0)
        # Update weights
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def fit(self, X, y, epochs=300, batch_size=128, verbose=True):
        history = []
        n = len(X)
        for epoch in range(1, epochs + 1):
            idx = np.random.permutation(n)
            epoch_loss = 0.0
            for start in range(0, n, batch_size):
                batch = idx[start:start + batch_size]
                xb, yb = X[batch], y[batch]
                probs = self.forward(xb)
                loss  = -np.log(probs[np.arange(len(yb)), yb] + 1e-9).mean()
                epoch_loss += loss * len(yb)
                self.backward(xb, yb)
            avg_loss = epoch_loss / n
            history.append(avg_loss)
            if verbose and epoch % 50 == 0:
                print(f"  Epoch {epoch:3d}/{epochs} — loss: {avg_loss:.4f}")
        return history

    def predict_proba(self, X):
        return self.forward(X)


#  Train the question-level MLP
print("Training MLP from scratch...")
mlp = MLP(n_in=15, n_hidden=64, n_out=5, lr=0.05, seed=SEED)
history = mlp.fit(X_train, y_train, epochs=300, batch_size=128, verbose=True)

#  Train the structural artifact ranker (pointwise: is THIS option the answer?)
print("\nTraining structural artifact ranker...")
struct_mlp = MLP(n_in=N_STRUCT_FEATS, n_hidden=64, n_out=2, lr=0.05, seed=SEED)
struct_history = struct_mlp.fit(S_train, sy_train, epochs=300, batch_size=256, verbose=False)
print(f"  final loss: {struct_history[-1]:.4f}")


def scores_struct(row, vec, mat, model, tfidf_s=None) -> np.ndarray:
    """P(correct) for each option from the structural ranker."""
    if tfidf_s is None:
        tfidf_s = [float(cosine_similarity(vec.transform([f"{row['prompt']} {row[o]}"]), mat).max())
                   for o in OPTIONS]
    F = structural_features(row, corpus_sims=tfidf_s)
    return model.predict_proba(F)[:, 1].astype(np.float32)


# Quick standalone check on the val split, split by question type
_struct_val = [scores_struct(row, tfidf_vectorizer, tfidf_matrix, struct_mlp)
               for _, row in val_split.iterrows()]
_struct_pred = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in _struct_val]
_ans = val_split["answer"].tolist()
_all = map_at_3(_ans, _struct_pred)
_hard = map_at_3([a for a, h in zip(_ans, val_hard_mask) if h],
                 [p for p, h in zip(_struct_pred, val_hard_mask) if h])
print(f"\nStructural ranker MAP@3 -- all val: {_all:.4f} | HARD val: {_hard:.4f}")
print(f"Reference points: chance 0.3667, constant 'B C A' 0.4213")
print("If HARD is meaningfully above 0.4213 the artifact signal is real and")
print("transfers to questions with no training near-duplicate.")

In [ ]:
# Evaluate on validation set 
proba_val = mlp.predict_proba(X_val)
mlp_preds = [[OPTIONS[j] for j in np.argsort(-proba_val[i])][:3]
             for i in range(len(X_val))]
mlp_map3  = map_at_3(val_split['answer'].tolist(), mlp_preds)
top1_mlp  = sum(p[0] == OPTIONS[y_val[i]] for i, p in enumerate(mlp_preds))

print(f"MAP@3 : {mlp_map3:.4f}")
print(f"Top-1 : {top1_mlp}/{len(val_split)} ({top1_mlp/len(val_split):.2%})")

# Log to W&B
wandb.init(
    project=WANDB_PROJECT,
    name="mlp-from-scratch",
    config={
        "model":        "MLP (pure NumPy)",
        "architecture": "15 → 64 → 5",
        "features":     "TF-IDF sim scores + pairwise diffs",
        "activation":   "ReLU + Softmax",
        "optimizer":    "mini-batch SGD",
        "epochs":       300,
        "batch_size":   128,
        "lr":           0.05,
        "val_size":     VAL_SIZE,
    },
    reinit=True
)
for i, loss in enumerate(history, 1):
    wandb.log({"epoch": i, "train_loss": loss})
_m = report_metrics("MLP (from scratch)", val_split["answer"].tolist(), mlp_preds)
wandb.log({f"val_{k}": v for k, v in _m.items()})
wandb.finish()
print("MLP run logged ✓")

## 11. LLM Reasoning Signal — bigger model + position debiasing

The leaderboard has sat in a 0.006 band across four very different ensembles.
That pattern fits a score made of two parts: questions with a near-duplicate in
the training data (answered almost perfectly by retrieval) and questions without
one (answered at close to chance). Every previous change improved only the first
group, which was already saturated -- hence no movement.

So this section targets the second group directly:

**A stronger base model.** Phi-3.5-mini is 3.8B (~69% MMLU). On genuinely novel
questions, base model capability is the binding constraint, so we move to a
larger instruct model in 4-bit (fits comfortably in 16GB) with automatic
fallback if it cannot be loaded.

**Full position-bias debiasing.** LLMs have a well-documented bias toward
particular answer *positions* regardless of content. Each question is scored
under all 5 cyclic rotations of the options, so every option appears in every
position exactly once and positional preference cancels out entirely. The logits
are mapped back to the original option order and averaged.


In [ ]:
import gc, sys, shutil, subprocess, importlib
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers

# ---------------------------------------------------------------------------
# 4-bit support. NOTE: `from transformers import BitsAndBytesConfig` succeeds
# even when the bitsandbytes LIBRARY is absent -- the config is just a dataclass
# in transformers. Checking that import therefore proves nothing, and every
# model then fails identically at load time. Import the real package instead.
# ---------------------------------------------------------------------------
def have_bnb() -> bool:
    try:
        import bitsandbytes  # noqa: F401
        return True
    except Exception:
        return False


if not have_bnb():
    print("bitsandbytes missing -> installing...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.46.1"],
        check=False,
    )
    importlib.invalidate_caches()

USE_4BIT = have_bnb()
print("4-bit quantization available:", USE_4BIT)

# transformers renamed torch_dtype -> dtype; pick whichever this version wants
try:
    from packaging.version import parse as _v
    DTYPE_KW = "dtype" if _v(transformers.__version__) >= _v("4.56.0") else "torch_dtype"
except Exception:
    DTYPE_KW = "torch_dtype"

free_gb = shutil.disk_usage("/").free / 1e9
print(f"transformers {transformers.__version__} (dtype kwarg: {DTYPE_KW}) | free disk: {free_gb:.0f}GB")


def make_bnb_config():
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )


# (model_id, want_4bit). Weights download at full precision before quantizing,
# so 14B needs ~30GB of disk even though it only occupies ~9GB of VRAM in 4-bit.
# It is skipped when disk is tight rather than failing after a long download.
CANDIDATES = []
if USE_4BIT and free_gb > 45:
    CANDIDATES.append(("Qwen/Qwen2.5-14B-Instruct", True))   # ~80% MMLU
if USE_4BIT:
    CANDIDATES.append(("Qwen/Qwen2.5-7B-Instruct", True))    # ~74% MMLU, ~15GB download
CANDIDATES += [
    ("Qwen/Qwen2.5-7B-Instruct", False),                     # fp16, needs a 16GB+ GPU
    ("microsoft/Phi-3.5-mini-instruct", True),
    ("microsoft/Phi-3.5-mini-instruct", False),              # ~7.6GB fp16, always fits
]

llm_model_id, llm_tokenizer, llm_model = None, None, None
for cand, want_4bit in CANDIDATES:
    if want_4bit and not USE_4BIT:
        continue
    try:
        print(f"\nLoading {cand} ({'4-bit' if want_4bit else 'fp16'}) ...")
        kwargs = {
            DTYPE_KW: torch.float16,
            "device_map": "auto",
            "trust_remote_code": True,
            "attn_implementation": "eager",
        }
        if want_4bit:
            kwargs["quantization_config"] = make_bnb_config()

        tok = AutoTokenizer.from_pretrained(cand, trust_remote_code=True)
        mdl = AutoModelForCausalLM.from_pretrained(cand, **kwargs)
        mdl.eval()
        llm_model_id, llm_tokenizer, llm_model = cand, tok, mdl
        break
    except Exception as e:
        print(f"  failed ({type(e).__name__}: {str(e)[:160]})")
        # release anything partially allocated before the next attempt
        for _name in ("mdl", "tok"):
            globals().pop(_name, None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

assert llm_model is not None, (
    "No LLM could be loaded. Check GPU is enabled and internet is on, "
    "or set CANDIDATES to a smaller model."
)

# from_pretrained() sets torch's GLOBAL default dtype while constructing a model
# and only restores it if construction succeeds. The fallback loop above can
# fail partway through (e.g. bitsandbytes missing), which leaves the default at
# float16 -- and every model built afterwards then silently gets fp16
# PARAMETERS. That is invisible until AMP training raises
# "ValueError: Attempting to unscale FP16 gradients." in Section 11b.
torch.set_default_dtype(torch.float32)
print(f"Default dtype restored to {torch.get_default_dtype()}")


def letter_token_ids(letter: str) -> list:
    """Token ids for a bare option letter, across surface variants.

    encode(" A")[0] returns the SentencePiece space marker, which is IDENTICAL
    for all five letters -- the original bug that made every question rank
    A,B,C,D,E. The LAST token of each variant is the real letter; keeping every
    variant and taking the max logit stays correct across tokenizers (Qwen's BPE
    treats "A" and " A" as two distinct tokens; Llama/Phi's SP does not).
    """
    ids = set()
    for variant in (letter, " " + letter):
        toks = llm_tokenizer.encode(variant, add_special_tokens=False)
        if toks:
            ids.add(toks[-1])
    return sorted(ids)


LETTER_TOKEN_IDS = {opt: letter_token_ids(opt) for opt in OPTIONS}

# Each option must own at least one token no other option claims, or the
# ranking is degenerate no matter how good the model is.
for o in OPTIONS:
    others = {t for p in OPTIONS if p != o for t in LETTER_TOKEN_IDS[p]}
    assert set(LETTER_TOKEN_IDS[o]) - others, f"option {o} has no distinct token: {LETTER_TOKEN_IDS}"

print(f"\nUsing: {llm_model_id}")
print(f"Device: {next(llm_model.parameters()).device}")
print(f"Letter token IDs: {LETTER_TOKEN_IDS}")


In [ ]:
N_FEWSHOT = 2       # kept small: prompt length drives runtime per rotation

# Rotations trade runtime directly against position-bias cancellation. The
# 5-fold MCQ training added in Section 11b is a far better use of the same GPU
# time than 5x LLM inference on a signal that has never earned significant
# ensemble weight, so this defaults to 1. Raise it to 5 for full debiasing if
# the session has time to spare (cost scales linearly).
N_PERM    = 1


def format_mcq(row, perm=None, answer_letter=None) -> str:
    """Render a question. `perm[i]` is the ORIGINAL option shown at position i."""
    if perm is None:
        perm = list(range(len(OPTIONS)))

    lines = ["Question: " + str(row["prompt"])]
    for i, orig in enumerate(perm):
        lines.append(f"{OPTIONS[i]}) {row[OPTIONS[orig]]}")
    lines.append("Answer:")

    text = "\n".join(lines)
    if answer_letter is not None:
        text += " " + str(answer_letter) + "\n"
    return text


def build_fewshot_prefix(row, shot_df, shot_prompt_embs, k: int = N_FEWSHOT) -> str:
    """The k most similar TRAIN questions, as in-context examples."""
    if k <= 0 or shot_df is None:
        return ""

    q = normalize(st_model.encode([str(row["prompt"])],
                                  convert_to_numpy=True, show_progress_bar=False))
    idx = np.argsort(-(q @ shot_prompt_embs.T)[0])[:k]

    # least similar first, so the closest example sits nearest the question
    return "".join(
        format_mcq(shot_df.iloc[i], answer_letter=shot_df.iloc[i]["answer"]) + "\n"
        for i in idx[::-1]
    )


@torch.no_grad()
def _letter_logits(text: str) -> np.ndarray:
    ids = llm_tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=3072).input_ids.to(llm_model.device)
    lg = llm_model(ids, use_cache=False).logits[0, -1, :].float()
    return np.array(
        [max(lg[t].item() for t in LETTER_TOKEN_IDS[o]) for o in OPTIONS],
        dtype=np.float64,
    )


def scores_llm(row, shot_df=None, shot_prompt_embs=None, n_perm: int = N_PERM) -> np.ndarray:
    """Position-debiased option scores.

    Averaging over all 5 cyclic rotations puts each option in each slot exactly
    once, so any preference the model has for a *position* cancels and only
    preference for *content* survives.
    """
    prefix = build_fewshot_prefix(row, shot_df, shot_prompt_embs)
    acc = np.zeros(len(OPTIONS))

    for p in range(n_perm):
        perm = [(i + p) % len(OPTIONS) for i in range(len(OPTIONS))]
        lg = _letter_logits(prefix + format_mcq(row, perm=perm))
        # position i displayed original option perm[i], so credit it there
        for i, orig in enumerate(perm):
            acc[orig] += lg[i]

    return (acc / n_perm).astype(np.float32)


In [ ]:
print(f"Evaluating {llm_model_id} on validation ({N_PERM} rotations/question)...")

llm_scores_val = []
for n, (_, row) in enumerate(val_split.iterrows(), 1):
    llm_scores_val.append(scores_llm(row, train_split, rag_prompt_embs))
    if n % 50 == 0:
        print(f"  {n}/{len(val_split)}")

val_answers_all = val_split["answer"].tolist()
llm_preds = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in llm_scores_val]

llm_map3 = map_at_3(val_answers_all, llm_preds)
llm_map3_hard = map_at_3(
    [a for a, h in zip(val_answers_all, val_hard_mask) if h],
    [p for p, h in zip(llm_preds, val_hard_mask) if h],
)
top1_llm = sum(p[0] == a for p, a in zip(llm_preds, val_answers_all))

print(f"\nLLM MAP@3 (all val)  : {llm_map3:.4f}")
print(f"LLM MAP@3 (HARD val) : {llm_map3_hard:.4f}   <-- the number that matters")
print(f"Top-1 : {top1_llm}/{len(val_split)} ({top1_llm/len(val_split):.2%})")
print(f"\nChance MAP@3 is 0.3667. If the HARD figure is not clearly above it,")
print(f"the base model cannot reason about these questions and a larger model")
print(f"is the only lever left.")

# first-choice spread: a position-biased model collapses onto one letter
from collections import Counter
print("\nLLM first-choice distribution:", dict(sorted(Counter(p[0] for p in llm_preds).items())))


In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="llm-debiased",
    config={
        "model": llm_model_id,
        "approach": f"{N_FEWSHOT}-shot retrieval + {N_PERM}-rotation position debiasing",
        "val_size": VAL_SIZE,
    },
    reinit=True,
)
_m = report_metrics("LLM", val_answers_all, llm_preds,
                    extra={"map3_hard": llm_map3_hard})
wandb.log({f"val_{k}": v for k, v in _m.items()})
wandb.finish()
print("LLM results logged.")


## 11b. Fine-tuned MCQ cross-encoder — 5-fold, out-of-fold

Every zero-shot and lookup signal has now been tried, and the leaderboard has not
moved outside ±0.005 across six very different ensembles. The one component that
actually *learns the task* is this one — and until now it was the worst-resourced
model in the notebook:

- It trained on `train_split` only (~1594 rows) while every weak signal was
  retrained on all 2000 rows in Section 14. The strongest model was using the
  least data.
- A single model was used at test time, where an ensemble of folds is strictly
  better for the same training budget.
- Ensemble weights were tuned on 406 validation rows with 9 free parameters and
  1200 random candidates — heavy enough selection to fit noise.

**5-fold GroupKFold (grouped by prompt, so near-duplicates cannot straddle a
fold) fixes all three.** Each fold trains on 80% and predicts the held-out 20%,
so every training row receives an honest out-of-fold (OOF) prediction and the
whole 2000 rows become available for weight tuning. At test time the five fold
models are averaged, which is the single most reliable source of gain in this
setup. Each fold scores the test set immediately after training and is then
freed, so only one model is resident at a time.


In [ ]:
import gc
from torch.utils.data import Dataset, DataLoader

try:
    import sentencepiece  # noqa: F401  (needed by the DeBERTa-v3 tokenizer)
except Exception:
    print("Installing sentencepiece...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=False)
    importlib.invalidate_caches()

from transformers import AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import GroupKFold

MCQ_CANDIDATES = ["microsoft/deberta-v3-large", "microsoft/deberta-v3-base"]
MCQ_MAX_LEN    = 256
MCQ_EPOCHS     = 2          # 2 x 5 folds sees more data than the old 3 x 1 run
MCQ_BATCH      = 2
MCQ_ACCUM      = 4          # effective batch 8
MCQ_LR         = 8e-6       # deberta-v3-large is unstable much above 1e-5
N_FOLDS        = 5          # drop to 3 if the session is short on time

# AMP requires fp32 PARAMETERS (autocast casts activations, not weights). A
# failed load in Section 11 can leave torch's global default at fp16, which the
# model would silently inherit -- surfacing later as
# "ValueError: Attempting to unscale FP16 gradients."
torch.set_default_dtype(torch.float32)

# Pick the backbone once, then rebuild it fresh for each fold.
mcq_model_id, mcq_tokenizer = None, None
for cand in MCQ_CANDIDATES:
    try:
        print(f"Resolving {cand} ...")
        mcq_tokenizer = AutoTokenizer.from_pretrained(cand)
        _probe = AutoModelForMultipleChoice.from_pretrained(
            cand, **{(DTYPE_KW if "DTYPE_KW" in globals() else "torch_dtype"): torch.float32})
        del _probe
        gc.collect()
        mcq_model_id = cand
        break
    except Exception as e:
        print(f"  failed ({type(e).__name__}: {str(e)[:140]})")
        gc.collect()

assert mcq_model_id is not None, "no multiple-choice backbone could be loaded"

mcq_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def new_mcq_model():
    """A freshly initialised model for one fold, guaranteed fp32."""
    _dk = DTYPE_KW if "DTYPE_KW" in globals() else "torch_dtype"
    m = AutoModelForMultipleChoice.from_pretrained(mcq_model_id, **{_dk: torch.float32})
    m = m.float().to(mcq_device)
    bad = [n for n, p in m.named_parameters() if p.dtype != torch.float32]
    assert not bad, f"non-fp32 parameter {bad[0]} -- AMP would fail on unscale_"
    if hasattr(m, "gradient_checkpointing_enable"):
        try:
            m.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            m.gradient_checkpointing_enable()
    return m


def mcq_encode(row, max_len=MCQ_MAX_LEN):
    """Encode the 5 (question, option) pairs; each tensor is (5, L)."""
    firsts  = [str(row["prompt"])] * len(OPTIONS)
    seconds = [str(row[o]) for o in OPTIONS]
    return mcq_tokenizer(firsts, seconds, truncation=True, max_length=max_len,
                         padding="max_length", return_tensors="pt")


class MCQDataset(Dataset):
    def __init__(self, df, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.with_labels = with_labels
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        item = dict(mcq_encode(row))
        if self.with_labels:
            item["labels"] = torch.tensor(OPTIONS.index(row["answer"]), dtype=torch.long)
        return item


def mcq_collate(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


print(f"Backbone: {mcq_model_id} | device: {mcq_device} | folds: {N_FOLDS}")


In [ ]:
@torch.no_grad()
def mcq_predict(model, df, batch_size=8) -> np.ndarray:
    """Logits over the 5 options for every row of df -> (n, 5)."""
    model.eval()
    loader = DataLoader(MCQDataset(df, with_labels=False), batch_size=batch_size,
                        shuffle=False, collate_fn=mcq_collate)
    out = []
    for batch in loader:
        batch = {k: v.to(mcq_device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(**batch).logits
        out.append(logits.float().cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)


def train_one_fold(tr_df, fold_id):
    """Train a fresh model on tr_df. Returns the trained model."""
    model = new_mcq_model()
    loader = DataLoader(MCQDataset(tr_df), batch_size=MCQ_BATCH, shuffle=True,
                        collate_fn=mcq_collate, drop_last=True)

    opt = torch.optim.AdamW(model.parameters(), lr=MCQ_LR, weight_decay=0.01)
    total = max(1, (len(loader) // MCQ_ACCUM) * MCQ_EPOCHS)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    for epoch in range(MCQ_EPOCHS):
        model.train()
        running, seen = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for step, batch in enumerate(loader):
            batch = {k: v.to(mcq_device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = model(**batch).loss / MCQ_ACCUM
            scaler.scale(loss).backward()

            if (step + 1) % MCQ_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)

            running += loss.item() * MCQ_ACCUM
            seen += 1
            if (step + 1) % 400 == 0:
                print(f"    fold {fold_id} epoch {epoch+1} "
                      f"step {step+1}/{len(loader)} loss {running/seen:.4f}")
        print(f"    fold {fold_id} epoch {epoch+1} done | loss {running/max(seen,1):.4f}")
    return model


# ---------------------------------------------------------------------------
# 5-fold GroupKFold over ALL training rows. Grouping by prompt keeps duplicate
# and near-duplicate prompts inside a single fold, so a fold's held-out rows are
# never answerable by looking at its own training rows.
#
# Each fold scores the test set right after training and is then freed, so peak
# memory stays at one model rather than N_FOLDS.
# ---------------------------------------------------------------------------
groups = train_df_clean["prompt"].values
gkf = GroupKFold(n_splits=N_FOLDS)

oof_by_id = {}
test_logit_sum = np.zeros((len(test_df_clean), len(OPTIONS)), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_df_clean, groups=groups), 1):
    tr_df = train_df_clean.iloc[tr_idx]
    va_df = train_df_clean.iloc[va_idx]
    print(f"\n=== fold {fold}/{N_FOLDS} | train {len(tr_df)} | holdout {len(va_df)} ===")

    model = train_one_fold(tr_df, fold)

    # honest out-of-fold predictions for the held-out rows
    va_logits = mcq_predict(model, va_df)
    for rid, vec in zip(va_df["id"].tolist(), va_logits):
        oof_by_id[rid] = vec

    fold_map3 = map_at_3(
        va_df["answer"].tolist(),
        [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in va_logits],
    )
    print(f"  fold {fold} holdout MAP@3 : {fold_map3:.4f}")

    # accumulate this fold's view of the test set, then release the model
    test_logit_sum += mcq_predict(model, test_df_clean).astype(np.float64)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

mcq_test_scores = (test_logit_sum / N_FOLDS).astype(np.float32)
assert len(oof_by_id) == len(train_df_clean), \
    f"OOF covers {len(oof_by_id)} of {len(train_df_clean)} rows"

print(f"\nOOF predictions for all {len(oof_by_id)} training rows.")
print(f"Test scores averaged over {N_FOLDS} fold models.")


In [ ]:
# OOF scores for the validation rows, looked up by row id. These come from the
# fold model that did NOT train on them, so they remain an honest signal for
# ensemble weight tuning.
mcq_scores_val = [oof_by_id[rid] for rid in val_split["id"].tolist()]


def scores_mcq(row):
    """Row-level accessor. Every row the ensemble scores -- training or test --
    already has a precomputed value, so this is a lookup rather than a forward
    pass. It raises loudly instead of silently returning noise if that ever
    stops being true."""
    rid = row["id"]
    if rid in oof_by_id:
        return oof_by_id[rid]
    if rid in _test_id_to_pos:
        return mcq_test_scores[_test_id_to_pos[rid]]
    raise KeyError(f"no MCQ score for row id {rid}")


_test_id_to_pos = {rid: i for i, rid in enumerate(test_df_clean["id"].tolist())}

# ---------------------------------------------------------------------------
# OOF metrics over ALL training rows -- a far more stable estimate than the old
# single 406-row split, and the number to compare against future changes.
# ---------------------------------------------------------------------------
oof_matrix  = np.stack([oof_by_id[rid] for rid in train_df_clean["id"].tolist()])
oof_answers = train_df_clean["answer"].tolist()
oof_preds   = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in oof_matrix]

mcq_oof_map3 = map_at_3(oof_answers, oof_preds)
oof_top1 = sum(p[0] == a for p, a in zip(oof_preds, oof_answers))

# same metric restricted to the validation rows, for comparability with the
# other sections
mcq_map3      = map_at_3(val_split["answer"].tolist(),
                         [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in mcq_scores_val])
_vp = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in mcq_scores_val]
mcq_map3_hard = map_at_3([a for a, h in zip(val_split["answer"], val_hard_mask) if h],
                         [p for p, h in zip(_vp, val_hard_mask) if h]) \
                if val_hard_mask.sum() else float("nan")
mcq_map3_easy = map_at_3([a for a, h in zip(val_split["answer"], ~val_hard_mask) if h],
                         [p for p, h in zip(_vp, ~val_hard_mask) if h]) \
                if (~val_hard_mask).sum() else float("nan")

print(f"Fine-tuned MCQ ({mcq_model_id}, {N_FOLDS}-fold)")
print(f"  OOF MAP@3 over all {len(oof_answers)} rows : {mcq_oof_map3:.4f}   <-- most stable estimate")
print(f"  OOF Top-1                          : {oof_top1}/{len(oof_answers)} "
      f"({oof_top1/len(oof_answers):.2%})")
print(f"  MAP@3 on val rows only             : {mcq_map3:.4f}")
_f = lambda v: f"{v:.4f}" if np.isfinite(v) else "n/a"
print(f"    EASY {_f(mcq_map3_easy)} | HARD {_f(mcq_map3_hard)}   (chance 0.3667)")

from collections import Counter
print("  OOF first-choice spread:", dict(sorted(Counter(p[0] for p in oof_preds).items())))

wandb.init(project=WANDB_PROJECT, name="mcq-kfold",
           config={"model": mcq_model_id, "folds": N_FOLDS, "epochs": MCQ_EPOCHS,
                   "lr": MCQ_LR, "max_len": MCQ_MAX_LEN}, reinit=True)
_m = report_metrics("MCQ (OOF, all rows)", oof_answers, oof_preds)
wandb.log({**{f"oof_{k}": v for k, v in _m.items()}, "val_map3": mcq_map3})
wandb.finish()


## 11c. Answer-Artifact Discriminator (from scratch, NumPy)

Every model so far learns from **2000 examples** — one per question. But the
dataset actually contains **10,000 labelled (option, is_correct) pairs**: 2000
correct answers and 8000 distractors. Nothing has ever used them.

That matters because distractors in generated MCQ datasets carry systematic
artifacts. Correct answers tend to be more specific and hedged ("typically",
"generally"); distractors tend to be absolute ("always", "never", "all") and are
often lexical variations of one another, which leaves the correct answer as the
odd one out. **None of this requires the question to have a near-duplicate in
training** — which is precisely the subset where the retrieval models are
useless and where every point of remaining headroom sits.

Concretely, each option is scored on how it relates to the corpus of *known
correct answers* versus the corpus of *known distractors*, plus its centrality
among its own five options and its surface statistics. A two-layer MLP written
from scratch in NumPy (the same class used in Section 10) maps those features to
P(correct).

Leakage is handled by the fold structure: when scoring a held-out row, the answer
bank is built only from the training folds, so a row's own options are never in
the bank it is compared against.


In [ ]:
# ---------------------------------------------------------------------------
# Encode every option text once (train + test), cached by row id.
# ---------------------------------------------------------------------------
def encode_option_matrix(df):
    """-> dict: row id -> (5, D) normalized embeddings of that row's options."""
    texts, index = [], []
    for _, row in df.iterrows():
        for o in OPTIONS:
            texts.append(str(row[o]))
            index.append(row["id"])
    embs = normalize(st_model.encode(texts, batch_size=64,
                                     show_progress_bar=True, convert_to_numpy=True))
    out, k = {}, 0
    for rid in df["id"].tolist():
        out[rid] = embs[k:k + len(OPTIONS)]
        k += len(OPTIONS)
    return out


print("Encoding option texts (train)...")
OPT_EMB = encode_option_matrix(train_df_clean)
print("Encoding option texts (test)...")
OPT_EMB.update(encode_option_matrix(test_df_clean))
print(f"Cached option embeddings for {len(OPT_EMB)} rows")

ABSOLUTES = ("always", "never", "all ", "none", "only", "must", "every",
             "cannot", "impossible", "no ", "entirely", "completely")
HEDGES    = ("typically", "generally", "usually", "often", "may", "can ",
             "tends", "likely", "approximately", "about", "some", "most")


def build_answer_bank(df):
    """Embeddings of known-correct and known-incorrect option texts."""
    corr, inc = [], []
    for _, row in df.iterrows():
        E = OPT_EMB[row["id"]]
        ai = OPTIONS.index(row["answer"])
        for j in range(len(OPTIONS)):
            (corr if j == ai else inc).append(E[j])
    return np.stack(corr).astype(np.float32), np.stack(inc).astype(np.float32)


N_DISC_FEATS = 16


def disc_features(row, bank) -> np.ndarray:
    """(5, N_DISC_FEATS) features -- one row per option."""
    corr_bank, inc_bank = bank
    E = OPT_EMB[row["id"]]                       # (5, D)

    sim_c = E @ corr_bank.T                      # (5, n_correct)
    sim_i = E @ inc_bank.T                       # (5, n_incorrect)
    top_c = np.sort(sim_c, axis=1)[:, -5:]
    top_i = np.sort(sim_i, axis=1)[:, -5:]

    self_sim = E @ E.T                           # (5, 5) option-to-option
    np.fill_diagonal(self_sim, np.nan)
    centrality = np.nanmean(self_sim, axis=1)
    max_other  = np.nanmax(self_sim, axis=1)

    texts = [str(row[o]) for o in OPTIONS]
    lows  = [t.lower() for t in texts]
    wlen  = np.array([len(t.split()) for t in texts], dtype=np.float64)
    clen  = np.array([len(t) for t in texts], dtype=np.float64)
    absol = np.array([sum(w in t for w in ABSOLUTES) for t in lows], dtype=np.float64)
    hedge = np.array([sum(w in t for w in HEDGES) for t in lows], dtype=np.float64)

    rel = lambda a: a / (a.mean() + 1e-9)        # relative to this question
    ctr = lambda a: a - a.mean()                 # centered within this question

    feats = np.stack([
        top_c[:, -1],                 # best match among known-correct answers
        top_c.mean(axis=1),
        top_i[:, -1],                 # best match among known distractors
        top_i.mean(axis=1),
        top_c[:, -1] - top_i[:, -1],  # the key discriminative contrast
        top_c.mean(axis=1) - top_i.mean(axis=1),
        ctr(top_c[:, -1] - top_i[:, -1]),
        centrality,                   # distractors cluster; the answer stands apart
        ctr(centrality),
        max_other,
        rel(wlen),
        ctr(wlen),
        rel(clen),
        absol,
        hedge,
        ctr(absol - hedge),
    ], axis=1).astype(np.float32)

    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


def disc_dataset(df, bank):
    """Flatten to (n*5, F) features and (n*5,) binary labels."""
    X, y = [], []
    for _, row in df.iterrows():
        X.append(disc_features(row, bank))
        ai = OPTIONS.index(row["answer"])
        lab = np.zeros(len(OPTIONS), dtype=np.int64)
        lab[ai] = 1
        y.append(lab)
    return np.concatenate(X, axis=0), np.concatenate(y, axis=0)


print(f"Feature extractor ready ({N_DISC_FEATS} features per option)")


In [ ]:
# ---------------------------------------------------------------------------
# 5-fold OOF training, mirroring Section 11b so both models' OOF predictions
# cover the same 2000 rows and can be tuned together.
# ---------------------------------------------------------------------------
disc_oof_by_id = {}
disc_gkf = GroupKFold(n_splits=N_FOLDS)
disc_groups = train_df_clean["prompt"].values

for fold, (tr_idx, va_idx) in enumerate(disc_gkf.split(train_df_clean, groups=disc_groups), 1):
    tr_df = train_df_clean.iloc[tr_idx]
    va_df = train_df_clean.iloc[va_idx]

    # bank from TRAINING folds only -- a held-out row's own options are never
    # in the bank it is scored against
    bank = build_answer_bank(tr_df)

    Xtr, ytr = disc_dataset(tr_df, bank)
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-8
    Xtr = (Xtr - mu) / sd

    model = MLP(n_in=N_DISC_FEATS, n_hidden=64, n_out=2, lr=0.05, seed=SEED + fold)
    model.fit(Xtr, ytr, epochs=400, batch_size=256, verbose=False)

    for _, row in va_df.iterrows():
        F = (disc_features(row, bank) - mu) / sd
        disc_oof_by_id[row["id"]] = model.predict_proba(F)[:, 1].astype(np.float32)

    hold = map_at_3(
        va_df["answer"].tolist(),
        [[OPTIONS[i] for i in np.argsort(-disc_oof_by_id[r["id"]])][:3]
         for _, r in va_df.iterrows()],
    )
    print(f"  fold {fold}/{N_FOLDS}: holdout MAP@3 {hold:.4f}")

assert len(disc_oof_by_id) == len(train_df_clean)

# Final model on ALL training data, for scoring the test set
disc_bank_full = build_answer_bank(train_df_clean)
Xf, yf = disc_dataset(train_df_clean, disc_bank_full)
DISC_MU, DISC_SD = Xf.mean(axis=0), Xf.std(axis=0) + 1e-8
disc_model_full = MLP(n_in=N_DISC_FEATS, n_hidden=64, n_out=2, lr=0.05, seed=SEED)
disc_model_full.fit((Xf - DISC_MU) / DISC_SD, yf, epochs=400, batch_size=256, verbose=False)


def scores_disc(row) -> np.ndarray:
    rid = row["id"]
    if rid in disc_oof_by_id:                 # training row -> honest OOF value
        return disc_oof_by_id[rid]
    F = (disc_features(row, disc_bank_full) - DISC_MU) / DISC_SD
    return disc_model_full.predict_proba(F)[:, 1].astype(np.float32)


disc_scores_val = [disc_oof_by_id[rid] for rid in val_split["id"].tolist()]

disc_oof_mat = np.stack([disc_oof_by_id[rid] for rid in train_df_clean["id"].tolist()])
disc_oof_preds = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in disc_oof_mat]
disc_oof_map3 = map_at_3(train_df_clean["answer"].tolist(), disc_oof_preds)
disc_top1 = sum(p[0] == a for p, a in zip(disc_oof_preds, train_df_clean["answer"]))

_vp = [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in disc_scores_val]
disc_map3 = map_at_3(val_split["answer"].tolist(), _vp)
disc_hard = (map_at_3([a for a, h in zip(val_split["answer"], val_hard_mask) if h],
                      [p for p, h in zip(_vp, val_hard_mask) if h])
             if val_hard_mask.sum() else float("nan"))

print(f"\nAnswer-Artifact Discriminator (NumPy MLP, {N_FOLDS}-fold)")
print(f"  OOF MAP@3 over all {len(disc_oof_mat)} rows : {disc_oof_map3:.4f}")
print(f"  OOF Top-1 : {disc_top1}/{len(disc_oof_mat)} ({disc_top1/len(disc_oof_mat):.2%})")
_fmt = lambda v: f"{v:.4f}" if np.isfinite(v) else "n/a"
print(f"  val MAP@3 : {disc_map3:.4f} | HARD : {_fmt(disc_hard)}")
print(f"  (chance = 0.3667 -- anything meaningfully above it is question-independent")
print(f"   signal that transfers to questions with no near-duplicate in training)")

wandb.init(project=WANDB_PROJECT, name="artifact-discriminator",
           config={"features": N_DISC_FEATS, "folds": N_FOLDS,
                   "train_pairs": len(train_df_clean) * len(OPTIONS)}, reinit=True)
_m = report_metrics("Discriminator (OOF)", train_df_clean["answer"].tolist(), disc_oof_preds)
wandb.log({**{f"oof_{k}": v for k, v in _m.items()}, "val_map3": disc_map3})
wandb.finish()


## 12. Similarity-gated ensemble

A single weight vector cannot serve both question types. Most validation rows
have a near-duplicate in train, so a global tuner is dominated by rows where the
retrieval models are already near-perfect -- and it therefore assigns the LLM
almost no weight. That weighting is then applied to the novel test questions,
where retrieval is worthless and the LLM was the only thing that could have
helped. This is why four different ensembles all landed on ~0.74.

So two weight vectors are tuned instead:

- **`w_easy`** — for questions with a close training neighbour (retrieval leads).
- **`w_hard`** — for questions without one (reasoning leads).

At prediction time each row is routed by its own similarity to the training set,
a quantity computed identically for validation and test.

Normalization centers each model's scores within the question, then divides by a
**global** per-model spread. Per-question z-scoring would be wrong here: it forces
every question to contribute equal magnitude, erasing the confidence differences
that the fusion is meant to exploit.


In [ ]:
# ---------------------------------------------------------------------------
# Shared scorers: one definition, used by validation AND submission.
# ---------------------------------------------------------------------------

def row_option_embs(row) -> np.ndarray:
    texts = [f"{row['prompt']} {row[opt]}" for opt in OPTIONS]
    return normalize(st_model.encode(texts, convert_to_numpy=True, show_progress_bar=False))


def scores_tfidf(row, vec, mat) -> np.ndarray:
    return np.array(
        [float(cosine_similarity(vec.transform([f"{row['prompt']} {row[opt]}"]), mat).max())
         for opt in OPTIONS], dtype=np.float32)


def scores_st(row, corpus_embs, opt_embs=None) -> np.ndarray:
    if opt_embs is None:
        opt_embs = row_option_embs(row)
    return (opt_embs @ corpus_embs.T).max(axis=1).astype(np.float32)


def scores_rag(row, corpus_embs, prompt_embs, answer_texts, k=5, opt_embs=None) -> np.ndarray:
    if opt_embs is None:
        opt_embs = row_option_embs(row)
    q = normalize(st_model.encode([str(row["prompt"])],
                                  convert_to_numpy=True, show_progress_bar=False))
    idx = np.argsort(-(q @ prompt_embs.T)[0])[:k]
    # answer text only -- matches the Section 9 definition
    ctx = normalize(st_model.encode([answer_texts[i] for i in idx],
                                    convert_to_numpy=True, show_progress_bar=False))
    return (0.6 * (opt_embs @ corpus_embs.T).max(axis=1)
            + 0.4 * (opt_embs @ ctx.T).max(axis=1)).astype(np.float32)


def scores_mlp(row, vec, mat, model, tfidf_s=None) -> np.ndarray:
    if tfidf_s is None:
        tfidf_s = scores_tfidf(row, vec, mat)
    sims = list(tfidf_s)
    diffs = [sims[i] - sims[j] for i in range(5) for j in range(i + 1, 5)]
    return model.predict_proba(
        np.array(sims + diffs, dtype=np.float32).reshape(1, -1))[0].astype(np.float32)


MODEL_NAMES = ["tfidf", "st", "rag", "mlp", "nli", "llm", "struct", "prior", "mcq", "disc"]


def all_scores(row, vec, mat, corpus_embs, prompt_embs, answer_texts,
               mlp_obj, shot_df, shot_prompt_embs,
               struct_obj, logprior,
               llm_override=None, mcq_override=None, disc_override=None) -> dict:
    """The *_override args reuse scores already computed in Sections 11 / 11b
    rather than recomputing them (the LLM costs N_PERM forward passes per row
    and is by far the slowest component)."""
    opt_embs = row_option_embs(row)
    tfidf_s  = scores_tfidf(row, vec, mat)
    return {
        "tfidf":  tfidf_s,
        "st":     scores_st(row, corpus_embs, opt_embs),
        "rag":    scores_rag(row, corpus_embs, prompt_embs, answer_texts, opt_embs=opt_embs),
        "mlp":    scores_mlp(row, vec, mat, mlp_obj, tfidf_s=tfidf_s),
        "nli":    scores_nli(row),
        "llm":    scores_llm(row, shot_df, shot_prompt_embs)
                  if llm_override is None else llm_override,
        "struct": scores_struct(row, vec, mat, struct_obj, tfidf_s=tfidf_s),
        "mcq":    scores_mcq(row) if mcq_override is None else mcq_override,
        "disc":   scores_disc(row) if disc_override is None else disc_override,
        # constant across questions -> acts as a tunable tie-break once centered
        "prior":  np.asarray(logprior, dtype=np.float32),
    }


def fit_scales(score_dicts: list) -> dict:
    """Global per-model spread, measured on question-centered scores."""
    scales = {}
    for m in MODEL_NAMES:
        centered = np.concatenate([
            np.asarray(d[m], dtype=np.float64) - np.mean(d[m]) for d in score_dicts])
        sd = float(centered.std())
        scales[m] = sd if sd > 1e-8 else 1.0
    return scales


def norm_scores(score_dict: dict, scales: dict) -> dict:
    out = {}
    for m in MODEL_NAMES:
        a = np.asarray(score_dict[m], dtype=np.float64)
        out[m] = np.clip((a - a.mean()) / scales[m], -6.0, 6.0)
    return out


def fuse(norm_dict: dict, weights: dict) -> list:
    total = np.zeros(len(OPTIONS))
    for name, w in weights.items():
        if w:
            total += w * norm_dict[name]
    return [OPTIONS[i] for i in np.argsort(-total)]


# ---------------------------------------------------------------------------
# Score the validation set once
# ---------------------------------------------------------------------------
print("Pre-computing model scores on val...")
val_score_dicts = []
for n, (idx, row) in enumerate(val_split.iterrows()):
    val_score_dicts.append(all_scores(
        row, tfidf_vectorizer, tfidf_matrix, train_embeddings,
        rag_prompt_embs, rag_correct_texts, mlp, train_split, rag_prompt_embs,
        struct_mlp, LABEL_LOGPRIOR,
        llm_override=llm_scores_val[n],    # reuse Section 11
        mcq_override=mcq_scores_val[n],    # reuse Section 11b
        disc_override=disc_scores_val[n]))  # reuse Section 11c
    if (n + 1) % 100 == 0:
        print(f"  {n+1}/{len(val_split)}")

val_answers = val_split["answer"].tolist()
val_scales  = fit_scales(val_score_dicts)
val_norm    = [norm_scores(d, val_scales) for d in val_score_dicts]
print("Global scales:", {k: round(v, 4) for k, v in val_scales.items()})


def eval_weights(weights, mask=None) -> float:
    """MAP@3 under `weights`. Returns nan for an empty selection, which callers
    must treat as 'no signal' rather than as a score."""
    idxs = list(range(len(val_norm))) if mask is None else list(np.where(mask)[0])
    if not idxs:
        return float("nan")
    return map_at_3([val_answers[i] for i in idxs],
                    [fuse(val_norm[i], weights)[:3] for i in idxs])


# Per-model MAP@3, split by question type. The HARD column is the one that
# predicts leaderboard movement; the EASY column is already saturated.
easy_mask = ~val_hard_mask
print(f"\n{'model':<10}{'EASY':>10}{'HARD':>10}   (chance = 0.3667)")
print("-" * 40)
solo = {}
for m in MODEL_NAMES:
    e, h = eval_weights({m: 1.0}, easy_mask), eval_weights({m: 1.0}, val_hard_mask)
    solo[m] = (e, h)
    fmt = lambda v: f"{v:>10.4f}" if np.isfinite(v) else f"{'n/a':>10}"
    print(f"{m:<10}{fmt(e)}{fmt(h)}")


MIN_TUNE_ROWS = 40   # below this, a tuned weight vector is fitting noise

# ---------------------------------------------------------------------------
# Baseline weightings.
#
# The label distribution is not uniform (B 24.5%, C 23.0%, A 18.5%, D 17.9%,
# E 16.2%), so ALWAYS answering "B C A" scores MAP@3 0.4213 -- well above the
# 0.3667 of a random ordering. On a question no model can actually answer, that
# prior is the best available strategy, and anything that produces a noisy
# ordering instead is throwing away 0.0546 per row.
#
# This matters because the lookup signals are near-perfect on questions with a
# training near-duplicate and pure noise on questions without one. A weighting
# tuned on validation (which is dominated by the former) is lookup-heavy, and
# applying it to the latter yields noise -- not the prior. So the prior-only
# vector below is used as an explicit FLOOR: no bucket is allowed to score worse
# than it.
# ---------------------------------------------------------------------------
LOOKUP_SIGNALS = ["tfidf", "st", "rag", "mlp", "struct"]

PRIOR_ONLY = {m: (1.0 if m == "prior" else 0.0) for m in MODEL_NAMES}

# For questions with no training neighbour, the lookup signals carry no
# information by construction, so this drops them and leans on the
# question-independent models plus the prior.
NO_LOOKUP = {m: 0.0 for m in MODEL_NAMES}
NO_LOOKUP.update({"prior": 3.0, "disc": 2.0, "mcq": 2.0, "llm": 1.0, "nli": 0.5})


def search_weights(mask, n_iter=1200, label="", fallback=None):
    if fallback is None:
        fallback = {m: 1.0 for m in MODEL_NAMES}
    rng = np.random.default_rng(SEED)
    GRID = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
    base = {m: 0.0 for m in MODEL_NAMES}
    cands = [
        {m: 1.0 for m in MODEL_NAMES},
        {**base, "tfidf": 1, "st": 1, "rag": 1, "mlp": 1, "nli": 0.5, "llm": 0.5},
        {**base, "nli": 1, "llm": 3, "struct": 2, "prior": 0.5},
        {**base, "tfidf": 0.5, "st": 0.5, "rag": 0.5, "mlp": 0.5, "nli": 1, "llm": 3, "struct": 2},
        {**base, "struct": 3, "prior": 1},                     # structure only
        {**base, "llm": 3, "struct": 3, "prior": 0.5},
        {**base, "prior": 1},                                  # prior-only floor
        {**base, "mcq": 3},                                    # fine-tuned model alone
        {**base, "mcq": 3, "llm": 1},
        {**base, "mcq": 3, "struct": 1, "prior": 0.5},
        {**base, "mcq": 3, "llm": 1.5, "struct": 1, "prior": 0.5},
        {**base, "disc": 3},                                   # artifact signal alone
        {**base, "mcq": 3, "disc": 3},                         # the two learned models
        {**base, "mcq": 3, "disc": 2, "prior": 0.5},
        {**base, "mcq": 2, "disc": 2, "tfidf": 1, "st": 1},
        {**base, "mcq": 3, "disc": 3, "llm": 1, "tfidf": 1, "st": 1, "prior": 0.5},
        dict(PRIOR_ONLY),                                      # the floor itself
        dict(NO_LOOKUP),                                       # principled hard-row mix
        {**base, "prior": 3, "disc": 2},
        {**base, "prior": 3, "mcq": 2},
        {**base, "prior": 2, "disc": 2, "mcq": 2, "llm": 1},
        {**base, "prior": 3, "disc": 1, "mcq": 1, "nli": 0.5, "llm": 0.5},
        {**base, "tfidf": 1, "st": 1, "rag": 1, "mlp": 1, "mcq": 3},
        {**base, "tfidf": 0.5, "st": 0.5, "rag": 0.5, "mlp": 0.5,
                 "nli": 0.5, "llm": 1.5, "struct": 1, "prior": 0.5, "mcq": 3},
    ]
    cands += [{m: float(rng.choice(GRID)) for m in MODEL_NAMES} for _ in range(n_iter)]

    # Sparse candidates: most signals here are near-duplicates of each other, and
    # a dense 9-weight vector fitted on a few hundred rows mostly fits noise.
    # These zero out all but 2-3 signals, which generalizes better.
    for _ in range(n_iter // 2):
        keep = rng.choice(MODEL_NAMES, size=int(rng.integers(2, 4)), replace=False)
        cands.append({m: (float(rng.choice([1.0, 2.0, 3.0])) if m in keep else 0.0)
                      for m in MODEL_NAMES})

    n = int(mask.sum())
    # Tuning on a handful of rows fits noise, and on zero rows every candidate
    # scores nan -- which previously left best_w as None and crashed on
    # best_w.items(). Fall back to a supplied default instead.
    if n < MIN_TUNE_ROWS:
        print(f"{label:<6} n={n:<5} too few rows to tune -> using fallback weights")
        return dict(fallback), float("nan")

    best_w, best_s = None, -np.inf
    for w in cands:
        if sum(w.values()) == 0:
            continue
        sc = eval_weights(w, mask)
        if np.isfinite(sc) and sc > best_s:
            best_s, best_w = sc, w

    if best_w is None:                       # every candidate scored nan
        print(f"{label:<6} n={n:<5} no usable score -> using fallback weights")
        return dict(fallback), float("nan")

    # FLOOR: never ship a weighting that loses to always answering "B C A" on
    # this bucket. Without this guard a lookup-heavy vector gets applied to rows
    # where lookup is uninformative, producing a noise ordering (~0.3667) in
    # place of the prior's 0.4213.
    floor_s = eval_weights(PRIOR_ONLY, mask)
    if np.isfinite(floor_s) and floor_s > best_s:
        print(f"{label:<6} n={n:<5} MAP@3 {best_s:.4f} is BELOW the prior floor "
              f"{floor_s:.4f} -> using prior")
        return dict(PRIOR_ONLY), floor_s

    print(f"{label:<6} n={n:<5} MAP@3 {best_s:.4f} (prior floor {floor_s:.4f})  "
          f"weights { {k: round(v,1) for k,v in best_w.items() if v} }")
    return best_w, best_s


# Tune the global weighting FIRST: it is both the comparison baseline and the
# fallback for a bucket that is empty or too small to tune on.
print("\nTuning weights:")
w_global, s_global = search_weights(np.ones(len(val_norm), dtype=bool), label="ALL")
w_easy,   s_easy   = search_weights(easy_mask,     label="EASY", fallback=w_global)
# A hard bucket too small to tune must NOT inherit the lookup-heavy global
# weighting -- lookup is uninformative on exactly these rows by construction.
w_hard,   s_hard   = search_weights(val_hard_mask, label="HARD", fallback=NO_LOOKUP)


# ---------------------------------------------------------------------------
# Confidence routing.
#
# Similarity-to-training is only a PROXY for "can the models answer this". A
# question can have a close training neighbour and still be scored ambiguously,
# and the fused margin measures that directly. When the top two options are
# nearly tied, the ranking is effectively arbitrary and the label prior is the
# better bet -- the same 0.3667-vs-0.4213 argument as above, applied per row
# instead of per bucket.
#
# TAU is tuned on validation over a grid that includes 0.0, so if this helps
# nothing the tuning selects 0.0 and behaviour is unchanged.
# ---------------------------------------------------------------------------
def fused_margin(norm_dict, weights) -> float:
    total = np.zeros(len(OPTIONS))
    for m, w in weights.items():
        if w:
            total += w * norm_dict[m]
    srt = np.sort(total)[::-1]
    return float(srt[0] - srt[1])


def gated_predict(norm_dict, sim_to_train, tau=0.0) -> list:
    """Route by training-neighbour similarity, then by fused confidence."""
    w = w_easy if sim_to_train >= HARD_THRESHOLD else w_hard
    if tau > 0.0 and fused_margin(norm_dict, w) < tau:
        w = PRIOR_ONLY          # too close to call -> take the prior
    return fuse(norm_dict, w)[:3]


print("\nTuning confidence threshold TAU:")
TAU_GRID = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
best_tau, best_tau_score = 0.0, -np.inf
for tau in TAU_GRID:
    sc = map_at_3(val_answers,
                  [gated_predict(nd, sim, tau)
                   for nd, sim in zip(val_norm, val_sim_to_train)])
    n_routed = sum(1 for nd, sim in zip(val_norm, val_sim_to_train)
                   if tau > 0 and fused_margin(
                       nd, w_easy if sim >= HARD_THRESHOLD else w_hard) < tau)
    print(f"  TAU {tau:<5} MAP@3 {sc:.4f}  ({n_routed} rows sent to the prior)")
    if sc > best_tau_score:
        best_tau_score, best_tau = sc, tau

# Ten grid points scored on a few hundred rows will show spurious differences of
# a few thousandths. Only adopt a non-zero TAU if it clears a margin that a
# random split is unlikely to produce; otherwise stay at 0 (routing disabled).
TAU_MIN_GAIN = 0.005
base_score = map_at_3(val_answers,
                      [gated_predict(nd, sim, 0.0)
                       for nd, sim in zip(val_norm, val_sim_to_train)])

if best_tau > 0.0 and (best_tau_score - base_score) >= TAU_MIN_GAIN:
    TAU = best_tau
    print(f"\nSelected TAU = {TAU}: {base_score:.4f} -> {best_tau_score:.4f} "
          f"(+{best_tau_score - base_score:.4f})")
else:
    TAU = 0.0
    print(f"\nTAU = 0.0 (best candidate {best_tau} gained only "
          f"{best_tau_score - base_score:+.4f}, under the {TAU_MIN_GAIN} margin "
          f"-> treated as noise, routing disabled)")


ens_predictions = [gated_predict(nd, s, TAU) for nd, s in zip(val_norm, val_sim_to_train)]
ens_map3 = map_at_3(val_answers, ens_predictions)
ens_map3_hard = map_at_3(
    [a for a, h in zip(val_answers, val_hard_mask) if h],
    [p for p, h in zip(ens_predictions, val_hard_mask) if h])

global_predictions = [fuse(nd, w_global)[:3] for nd in val_norm]
_prior_pred = [fuse(nd, PRIOR_ONLY)[:3] for nd in val_norm]
print(f"Prior-only       MAP@3 (all) : {map_at_3(val_answers, _prior_pred):.4f}"
      f"   <-- trivial baseline the ensemble must beat")
print(f"\nGated ensemble   MAP@3 (all) : {ens_map3:.4f} | (HARD) : {ens_map3_hard:.4f}")
print(f"Single-weight    MAP@3 (all) : {map_at_3(val_answers, global_predictions):.4f}")
print("\nIf gating does not beat the single weighting on HARD rows, the LLM is")
print("not adding usable signal and the base model is the bottleneck.")


In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="gated-ensemble",
    config={
        "model": "similarity-gated score fusion",
        "components": MODEL_NAMES,
        "llm": llm_model_id,
        "w_easy": {k: float(v) for k, v in w_easy.items()},
        "w_hard": {k: float(v) for k, v in w_hard.items()},
        "hard_threshold": HARD_THRESHOLD,
    },
    reinit=True,
)
_m = report_metrics("Gated ensemble", val_answers, ens_predictions,
                    extra={"map3_hard": ens_map3_hard, "map3_easy": s_easy})
wandb.log({f"val_{k}": v for k, v in _m.items()})

# Per-class breakdown: macro F1 can hide a model that scores well by favouring
# the frequent letters (B, C) while failing on the rare ones (D, E).
print("\nPer-class report (top-1):")
print(classification_report(val_answers, [p[0] for p in ens_predictions],
                            labels=OPTIONS, zero_division=0))
wandb.finish()


## 13. Model Comparison


In [ ]:
print(f"{'Model':<26}{'EASY':>10}{'HARD':>10}")
print("-" * 46)
_f = lambda v: f"{v:>10.4f}" if np.isfinite(v) else f"{'n/a':>10}"
for m in MODEL_NAMES:
    e, h = solo[m]
    print(f"{m:<26}{_f(e)}{_f(h)}")
print("-" * 46)
print(f"{'Gated ensemble':<26}{_f(s_easy)}{_f(ens_map3_hard)}")
print(f"\nChance = 0.3667. The fine-tuned MCQ model is the only signal trained")
print(f"on the task itself, so its HARD column is the key number to watch.")
print(f"EASY rows are saturated; every point of remaining")
print(f"headroom is in the HARD column.")


## 14. Generate Submission


In [ ]:
# Every test-time index is built from train_df_clean (NOT raw train_df): test
# rows are cleaned, so an uncleaned index would leave boilerplate on one side of
# every similarity computation.
print("Training TF-IDF on full data...")
full_correct_texts = [str(row["prompt"]) + " " + str(row[row["answer"]])
                      for _, row in train_df_clean.iterrows()]
full_answer_texts  = [str(row[row["answer"]]) for _, row in train_df_clean.iterrows()]

# Transductive vocabulary: fit the vectorizer on train AND test text, then index
# only the training rows. Test questions use vocabulary that never appears in
# training, and any such term was previously dropped from the vector space
# entirely -- so those options were compared on their remaining words alone.
# Fitting on both sides uses no test labels, so it leaks nothing.
tfidf_fit_corpus = full_correct_texts + [
    str(row["prompt"]) + " " + " ".join(str(row[o]) for o in OPTIONS)
    for _, row in test_df_clean.iterrows()
]
tfidf_vectorizer_full = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True,
    max_features=80000, strip_accents="unicode", min_df=1)
tfidf_vectorizer_full.fit(tfidf_fit_corpus)
tfidf_matrix_full = tfidf_vectorizer_full.transform(full_correct_texts)
print(f"  vocabulary {len(tfidf_vectorizer_full.vocabulary_)} "
      f"(fitted on train+test text, indexed on train only)")

print("Encoding sentence embeddings on full data...")
train_embeddings_full = normalize(st_model.encode(
    full_correct_texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True))

print("Building RAG / few-shot index on full data...")
rag_prompt_embs_full = normalize(st_model.encode(
    train_df_clean["prompt"].tolist(), batch_size=32,
    show_progress_bar=False, convert_to_numpy=True))

print("Extracting MLP features on full data...")
X_full, y_full = extract_features(train_df_clean, tfidf_vectorizer_full, tfidf_matrix_full)

print("Training MLP on full data...")
mlp_full = MLP(n_in=15, n_hidden=64, n_out=5, lr=0.05, seed=SEED)
mlp_full.fit(X_full, y_full, epochs=300, batch_size=128, verbose=False)

print("Training structural artifact ranker on full data...")
S_full, sy_full = build_struct_dataset(train_df_clean, tfidf_vectorizer_full, tfidf_matrix_full)
struct_mlp_full = MLP(n_in=N_STRUCT_FEATS, n_hidden=64, n_out=2, lr=0.05, seed=SEED)
struct_mlp_full.fit(S_full, sy_full, epochs=300, batch_size=256, verbose=False)

# label prior from ALL training rows for test-time use
LABEL_LOGPRIOR_FULL = label_log_prior(train_df_clean)
print(f"  full-data label prior: {dict(zip(OPTIONS, np.round(np.exp(LABEL_LOGPRIOR_FULL), 3)))}")

# ---------------------------------------------------------------------------
# DIAGNOSTIC: how much of the TEST set can retrieval possibly answer?
# This is the number that explains the leaderboard plateau. If test questions
# sit much further from the training set than validation questions do, then the
# validation score was always going to overstate the leaderboard, and the test
# rows below the threshold are where the remaining points live.
# ---------------------------------------------------------------------------
test_sim_to_train = max_similarity_to(
    test_df_clean["prompt"].tolist(), train_df_clean["prompt"].tolist())

test_hard_mask = test_sim_to_train < HARD_THRESHOLD

print("\n" + "=" * 62)
print("SIMILARITY TO TRAINING SET  (>= %.2f == answerable by lookup)" % HARD_THRESHOLD)
print("=" * 62)
for name, sims in [("VALIDATION", val_sim_to_train), ("TEST", test_sim_to_train)]:
    frac_easy = float((sims >= HARD_THRESHOLD).mean())
    print(f"{name:<12} n={len(sims):<5} mean={sims.mean():.3f}  median={np.median(sims):.3f}  "
          f"EASY={frac_easy:.1%}  HARD={1-frac_easy:.1%}")
print("=" * 62)
print(f"HARD test rows: {int(test_hard_mask.sum())}/{len(test_sim_to_train)} "
      f"-- these are routed through w_hard.")
print("If TEST is far more HARD than VALIDATION, that alone explains why every")
print("val-tuned ensemble so far converged to the same leaderboard score.")

print("\nAll models ready.")


In [ ]:
def test_score_dict(row) -> dict:
    """Identical computation to validation, with full-data indexes."""
    return all_scores(
        row,
        tfidf_vectorizer_full, tfidf_matrix_full,
        train_embeddings_full, rag_prompt_embs_full, full_answer_texts,
        mlp_full,
        train_df_clean, rag_prompt_embs_full,
        struct_mlp_full, LABEL_LOGPRIOR_FULL,
    )


In [ ]:
print("Scoring test set...")
test_score_dicts = []
for i, (_, row) in enumerate(test_df_clean.iterrows(), 1):
    test_score_dicts.append(test_score_dict(row))
    if i % 25 == 0:
        print(f"  {i}/{len(test_df_clean)}")

# Reuse the validation-fitted scales so test scores are calibrated exactly as
# the weights were tuned.
test_norm = [norm_scores(d, val_scales) for d in test_score_dicts]

# Route each row by its own similarity to the training set
rows = [
    {"ID": row["id"], "Prediction": " ".join(gated_predict(nd, sim, TAU))}
    for (_, row), nd, sim in zip(test_df_clean.iterrows(), test_norm, test_sim_to_train)
]
submission = pd.DataFrame(rows)

assert len(submission) == len(test_df_clean), "row count mismatch"
assert submission["Prediction"].str.split().apply(len).eq(3).all(), "need exactly 3 labels"
assert submission["Prediction"].apply(
    lambda p: len(set(p.split())) == 3 and set(p.split()) <= set(OPTIONS)
).all(), "labels must be 3 distinct values from A-E"

submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")
print(submission.head(10).to_string(index=False))

print(f"\nRouted via w_easy: {int((~test_hard_mask).sum())} rows")
print(f"Routed via w_hard: {int(test_hard_mask.sum())} rows")

# A collapsed first-choice distribution means a scorer went degenerate
print("\nFirst-choice distribution:")
print(submission["Prediction"].str[0].value_counts().sort_index())
